In [1]:
# Install (run once per session ideally)
!pip install opencv-python

import os
import cv2
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm

# Device configuration (VERY IMPORTANT)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
class SignLanguageDataset(Dataset):
    
    def __init__(self, root_dir, num_frames=32):  # 🔽 32 is a good balance
        self.samples = []
        self.total_videos = 0
        self.num_frames = num_frames
        self.bad_videos = 0
    
        # ✅ Improved transform (CRITICAL)
        self.transform = transforms.Compose([
            transforms.ToPILImage(),                     # required before resize
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(p=0.3),      # slight augmentation
            transforms.ToTensor(),
            transforms.Normalize(                        # 🔥 VERY IMPORTANT
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])
        
        for group in os.listdir(root_dir):
            group_path = os.path.join(root_dir, group)
            if not os.path.isdir(group_path):
                continue
            
            for inner in os.listdir(group_path):
                inner_path = os.path.join(group_path, inner)
                if not os.path.isdir(inner_path):
                    continue
                
                for person in os.listdir(inner_path):
                    person_path = os.path.join(inner_path, person)
                    if not os.path.isdir(person_path):
                        continue
                    
                    for sentence in os.listdir(person_path):
                        sentence_path = os.path.join(person_path, sentence)
                        if not os.path.isdir(sentence_path):
                            continue
                        
                        for file in os.listdir(sentence_path):
                            if file.endswith(('.mp4', '.avi', '.mov')):
                                video_path = os.path.join(sentence_path, file)
                                label = sentence.lower()
                                self.samples.append((video_path, label))
                                self.total_videos += 1
    
        # ✅ Stable label mapping
        all_labels = [s[1] for s in self.samples]
        self.labels = sorted(list(set(all_labels)))  # sorted = consistency
        self.label_map = {label: idx for idx, label in enumerate(self.labels)}
            
    def __len__(self):
        return len(self.samples)
    
    def load_video(self, path):
        cap = cv2.VideoCapture(path)
        frames = []
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        
        cap.release()
        
        # ❌ bad video case
        if len(frames) == 0:
            self.bad_videos += 1
            return torch.zeros((self.num_frames, 3, 224, 224))
        
        # ✅ pad if needed
        if len(frames) < self.num_frames:
            frames = frames + [frames[-1]] * (self.num_frames - len(frames))
        
        # ✅ uniform sampling (GOOD, keep this)
        idxs = np.linspace(0, len(frames)-1, self.num_frames).astype(int)
        frames = [frames[i] for i in idxs]
        
        # ✅ apply transform
        frames = [self.transform(frame) for frame in frames]
        
        return torch.stack(frames)
    
    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        
        frames = self.load_video(video_path)
        label = self.label_map[label]
        
        return frames, label

In [3]:
from torchvision.models import resnet50, ResNet50_Weights

class CNNLSTM(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        
        # ✅ Modern pretrained ResNet
        resnet = resnet50(weights=ResNet50_Weights.DEFAULT)
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])
        
        # ✅ Freeze CNN (important for small dataset)
        for param in self.cnn.parameters():
            param.requires_grad = False
        
        # ✅ Stronger BiLSTM
        self.lstm = nn.LSTM(
            input_size=2048,
            hidden_size=512,        # 🔥 increased
            num_layers=2,           # 🔥 deeper
            batch_first=True,
            bidirectional=True      # 🔥 key improvement
        )
        
        # ✅ Dropout for regularization
        self.dropout = nn.Dropout(0.5)
        
        # ✅ Adjust FC for BiLSTM (512 * 2)
        self.fc = nn.Linear(1024, num_classes)
    
    def forward(self, x):
        B, T, C, H, W = x.shape
        
        # Merge batch and time
        x = x.view(B * T, C, H, W)
        
        # CNN feature extraction
        features = self.cnn(x)                  # (B*T, 2048, 1, 1)
        features = features.view(B, T, 2048)    # (B, T, 2048)
        
        # LSTM
        lstm_out, _ = self.lstm(features)       # (B, T, 1024)
        
        # Use last timestep output (better than hidden state)
        out = lstm_out[:, -1, :]                # (B, 1024)
        
        out = self.dropout(out)
        out = self.fc(out)
        
        return out

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ Keep full dataset separate
full_dataset = SignLanguageDataset(
    "/kaggle/input/datasets/belovedorange/nlp-dataset",
    num_frames=32
)

# # ===== DEBUG SUBSET =====
# from torch.utils.data import Subset

# subset_size = int(0.2 * len(full_dataset))  # 🔥 20% instead of 1%
# indices = list(range(subset_size))

dataset = full_dataset

# ===== SPLIT =====
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print("Total videos found:", full_dataset.total_videos)
print(f"Using subset: {len(dataset)} samples")

# ===== DATALOADERS =====
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

# ✅ Correct number of classes
num_classes = len(full_dataset.labels)

# # ===== MODEL =====
# model = CNNLSTM(num_classes=num_classes).to(device)

# # ===== TRAINING SETUP =====
# criterion = nn.CrossEntropyLoss()

# optimizer = torch.optim.AdamW(   # 🔥 better than Adam
#     model.parameters(),
#     lr=1e-3,                    # 🔥 higher since CNN is frozen
#     weight_decay=1e-4
# )

Total videos found: 2485
Using subset: 2485 samples


In [5]:
# epochs = 10
# best_val_acc = 0
# patience = 3
# counter = 0

# # 🔥 Add scheduler
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#     optimizer,
#     mode='max',
#     factor=0.5,
#     patience=2
# )

# for epoch in range(epochs):
    
#     # ===== TRAIN =====
#     model.train()
#     train_loss = 0
#     train_correct = 0
#     train_total = 0
    
#     for videos, labels in tqdm(train_loader):
#         videos = videos.to(device)
#         labels = labels.to(device)
        
#         optimizer.zero_grad()
        
#         outputs = model(videos)
#         loss = criterion(outputs, labels)
        
#         loss.backward()
        
#         # 🔥 Gradient clipping (important for LSTM stability)
#         torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
#         optimizer.step()
        
#         train_loss += loss.item()
        
#         _, predicted = torch.max(outputs, 1)
#         train_total += labels.size(0)
#         train_correct += (predicted == labels).sum().item()
    
    
#     # ===== VALIDATION =====
#     model.eval()
#     val_correct = 0
#     val_total = 0
    
#     with torch.no_grad():
#         for videos, labels in val_loader:
#             videos = videos.to(device)
#             labels = labels.to(device)
            
#             outputs = model(videos)
            
#             _, predicted = torch.max(outputs, 1)
#             val_total += labels.size(0)
#             val_correct += (predicted == labels).sum().item()
    
    
#     val_acc = (val_correct / val_total) * 100
    
#     # 🔥 Scheduler step
#     scheduler.step(val_acc)
    
#     # ===== EARLY STOPPING =====
#     if val_acc > best_val_acc:
#         best_val_acc = val_acc
#         counter = 0
        
#         torch.save(model.state_dict(), "best_model.pth")
#         print("✅ Best model saved!")
#     else:
#         counter += 1
#         print(f"No improvement. Counter: {counter}/{patience}")
    
#     if counter >= patience:
#         print("⛔ Early stopping triggered")
#         break
    
    
#     # ===== PRINT =====
#     print(f"\nEpoch {epoch+1}")
#     print(f"Train Loss: {train_loss / len(train_loader):.4f}")
#     print(f"Train Accuracy: {(train_correct / train_total) * 100:.2f}%")
#     print(f"Val Accuracy: {val_acc:.2f}%")


# print("\n===== DATASET STATS =====")
# print("Total videos:", full_dataset.total_videos)
# print("Bad videos encountered:", full_dataset.bad_videos)
# print("Total classes:", len(full_dataset.labels))
# print("Sample labels:", full_dataset.labels[:5])

# if full_dataset.total_videos > 0:
#     print("Percentage discarded:", 
#           (full_dataset.bad_videos / full_dataset.total_videos) * 100, "%")

In [6]:
# ===== MODEL =====
model = CNNLSTM(num_classes=num_classes).to(device)

# 🔥 LOAD PREVIOUS MODEL
model.load_state_dict(torch.load(
    "/kaggle/input/notebooks/drishtiaggarwal6284/cnn-bilstm-based-csl-model/best_model.pth",
    map_location=device
))
print("✅ Loaded previous trained model")


# ===== TRAINING SETUP =====
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,   # SAME as before
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=2
)

# 🔥 RESUME SETTINGS
start_epoch = 9          # last completed epoch
epochs = 15              # total epochs you want now
best_val_acc = 50.50     # from logs
patience = 3
counter = 0


# ===== TRAIN LOOP =====
for epoch in range(start_epoch, epochs):
    
    print(f"\n🚀 Resuming Epoch {epoch+1}")
    
    # ===== TRAIN =====
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for videos, labels in tqdm(train_loader):
        videos = videos.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
        
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
    
    
    # ===== VALIDATION =====
    model.eval()
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for videos, labels in val_loader:
            videos = videos.to(device)
            labels = labels.to(device)
            
            outputs = model(videos)
            _, predicted = torch.max(outputs, 1)
            
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    
    
    val_acc = (val_correct / val_total) * 100
    
    scheduler.step(val_acc)
    
    
    # ===== EARLY STOPPING =====
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        counter = 0
        
        torch.save(model.state_dict(), "continued_best_model.pth")
        print("✅ Improved model saved!")
    else:
        counter += 1
        print(f"No improvement. Counter: {counter}/{patience}")
    
    if counter >= patience:
        print("⛔ Early stopping triggered")
        break
    
    
    print(f"Train Loss: {train_loss / len(train_loader):.4f}")
    print(f"Train Accuracy: {(train_correct / train_total) * 100:.2f}%")
    print(f"Val Accuracy: {val_acc:.2f}%")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 212MB/s]


✅ Loaded previous trained model

🚀 Resuming Epoch 10


100%|██████████| 249/249 [1:01:54<00:00, 14.92s/it]


✅ Improved model saved!
Train Loss: 1.1468
Train Accuracy: 61.37%
Val Accuracy: 64.99%

🚀 Resuming Epoch 11


...

✅ Improved model saved!
Train Loss: 0.7687
Train Accuracy: 74.20%
Val Accuracy: 68.61%

🚀 Resuming Epoch 14


100%|██████████| 249/249 [1:00:21<00:00, 14.54s/it]


✅ Improved model saved!
Train Loss: 0.6852
Train Accuracy: 76.21%
Val Accuracy: 71.63%

🚀 Resuming Epoch 15


100%|██████████| 249/249 [1:00:25<00:00, 14.56s/it]


No improvement. Counter: 1/3
Train Loss: 0.6687
Train Accuracy: 78.22%
Val Accuracy: 70.42%
